# Binary Fault Classification — PINN Algo (PaperFeatureSelector)

This notebook trains the binary classifier using the reference paper's own feature-selection method (Li et al., EUSIPCO 2025, "Algorithm 1"): physics-defined candidate frequency bins (built from BPFO/BPFI/BSF harmonics and shaft-frequency sidebands) are z-scored per class and kept where the one-vs-rest z-score difference exceeds a threshold `tau`, rather than using this project's ~36 hand-engineered features (`PhysicsFeatureExtractor`).

It uses the same Paderborn windows, grouped split (`stratified_group_split(seed=17)`), and class-weighting formula as `Binary_Classification_Baseline.ipynb` and `Binary_Classification_PhysicsWeight.ipynb`, so the three notebooks are directly comparable. The controlled differences here are the feature set (`PaperFeatureSelector` instead of `PhysicsFeatureExtractor`) and `use_physics_loss=False` / `physics_weight=0.0` -- the paper's own training objective is plain cross-entropy; its "physics-informed" contribution is the feature *selection*, not a loss term.

**This is a different mechanism from `Binary_Classification_PhysicsWeight.ipynb`** (formerly misnamed `Binary_Classification_PINN_Algo.ipynb`), which tests this project's own physics-consistency loss term and has nothing to do with the reference paper's algorithm. See `model.py`'s and `paper_features.py`'s module docstrings for the full disambiguation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
!mkdir -p /kaggle/working/repo
!cp -r "/kaggle/input/datasets/anaranyosarkar27/pinn-motor-fault-code/induction-motor-fault-classification-via-PINN" /kaggle/working/repo/project
!pip install -e /kaggle/working/repo/project --break-system-packages -q

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/project/src")
import pinn_motor_fault
print("Package imported successfully")

In [ ]:
from pinn_motor_fault.fast_train import compute_class_weights, fast_fit
from pinn_motor_fault.paper_features import PaperFeatureSelector
from pinn_motor_fault.features import PhysicsFeatureExtractor
from pinn_motor_fault.model import PhysicsInformedNN
from pinn_motor_fault.paderborn import load_paderborn_windows
from pinn_motor_fault.results import write_evaluation_artifacts
from pinn_motor_fault.train import stratified_group_split

DATA_DIR = Path("/kaggle/input/datasets/dippatel03/paderborn-db")
MODEL_PATH = Path("/kaggle/working/models/paderborn_binary_paper.npz")
RESULTS_DIR = Path("/kaggle/working/reports/results_binary_paper")
BINARY_CLASS_NAMES = ("healthy", "faulty")

windows, original_labels, sources = load_paderborn_windows(
    data_dir=DATA_DIR,
    window_size=4096,
    stride=2048,
    signal_preference="vibration",
    max_windows_per_file=30,
)
labels = np.where(original_labels == "healthy", "healthy", "faulty")
print("Original labels:", dict(zip(*np.unique(original_labels, return_counts=True))))
print("Binary labels:", dict(zip(*np.unique(labels, return_counts=True))))

# Split BEFORE fitting the feature selector, so tau-threshold bin selection is
# computed only from training data (avoids leaking test windows into the
# choice of which frequency bins get selected).
train_idx, test_idx = stratified_group_split(labels, sources, test_fraction=0.25, seed=17)

# Reference-paper Algorithm 1 feature selection: fit on the training split only.
selector = PaperFeatureSelector(tau=1.0, sample_rate_hz=64_000.0)
selector.fit(windows[train_idx], labels[train_idx])
features = selector.transform(windows)
feature_names = selector.feature_names()
print(f"Selected {features.shape[1]} paper-algorithm frequency-bin features (tau={selector.tau})")

# PhysicsInformedNN.fit()/fast_fit() always require a physics_targets array in
# their loss/gradient computation, even when use_physics_loss=False (the flag
# only controls whether the physics term is ADDED to the total loss/gradient
# -- see model.py's _evaluate_indices/_train_batch). Reuse
# PhysicsFeatureExtractor purely for its physics_targets output here; its
# 36 engineered features are discarded since this notebook uses the paper's
# frequency-bin features instead.
diagnostic_batch = PhysicsFeatureExtractor().transform(windows, sources)
binary_physics_targets = np.column_stack([
    diagnostic_batch.physics_targets[:, 0],
    diagnostic_batch.physics_targets[:, 1:].sum(axis=1),
])

class_weights = compute_class_weights(labels[train_idx], BINARY_CLASS_NAMES)
print("Train/test windows:", train_idx.size, test_idx.size)
print("Class weights:", class_weights)

In [ ]:
# Paper-algorithm classifier: PaperFeatureSelector features, physics-consistency
# loss term disabled (matches the reference paper's own cross-entropy-only
# training objective -- the paper's "physics-informed" part is the feature
# selection above, not a loss term).
model = PhysicsInformedNN(
    input_dim=features.shape[1],
    hidden_dim=48,
    physics_weight=0.0,
    learning_rate=0.1,
    use_physics_loss=False,
    class_names=BINARY_CLASS_NAMES,
)
history = fast_fit(
    model,
    features[train_idx],
    labels[train_idx],
    binary_physics_targets[train_idx],
    features[test_idx],
    labels[test_idx],
    binary_physics_targets[test_idx],
    epochs=60,
    batch_size=64,
    class_weights=class_weights,
)
model.save(MODEL_PATH)
print(f"Saved paper-algorithm model: {MODEL_PATH}")

In [ ]:
test_predictions = model.predict(features[test_idx])
test_probabilities = model.predict_proba(features[test_idx])
train_loss, train_accuracy = model.loss_and_accuracy(
    features[train_idx], labels[train_idx], binary_physics_targets[train_idx]
)
test_loss, test_accuracy = model.loss_and_accuracy(
    features[test_idx], labels[test_idx], binary_physics_targets[test_idx]
)
settings = {
    "model_path": str(MODEL_PATH),
    "data_dir": str(DATA_DIR),
    "sample_count": int(test_idx.size),
    "train_sample_count": int(train_idx.size),
    "test_sample_count": int(test_idx.size),
    "unique_train_files": int(len(set(sources[index] for index in train_idx))),
    "unique_test_files": int(len(set(sources[index] for index in test_idx))),
    "window_size": 4096,
    "stride": 2048,
    "signal": "vibration",
    "split_strategy": "stratified_group_by_mat_file",
    "feature_set": f"PaperFeatureSelector tau={selector.tau} frequency-bin features",
    "algorithm": "reference-paper Algorithm 1 feature selection, plain cross-entropy",
    "use_physics_loss": False,
    "epochs": 60,
    "max_windows_per_file": 30,
    "physics_weight": 0.0,
    "learning_rate": 0.1,
    "train_loss": float(train_loss),
    "test_loss": float(test_loss),
    "class_weights": class_weights,
}
metrics = write_evaluation_artifacts(
    output_dir=RESULTS_DIR,
    labels=labels[test_idx],
    predictions=test_predictions,
    probabilities=test_probabilities,
    sources=[sources[index] for index in test_idx],
    model=model,
    feature_names=feature_names,
    windows=windows[test_idx],
    signal_name="vibration",
    settings=settings,
    training_history=history,
)
test_X = features[test_idx]
test_y = labels[test_idx]
results_dir = str(RESULTS_DIR)
print(f"Paper-algorithm test accuracy: {test_accuracy:.4f}")
print(f"Saved results: {RESULTS_DIR}")

In [ ]:
import json
from IPython.display import SVG, display

print(pd.read_csv(f"{results_dir}/confusion_matrix.csv", index_col=0))
metrics = json.load(open(f"{results_dir}/metrics.json"))
print(json.dumps(metrics["class_metrics"], indent=2))
display(SVG(f"{results_dir}/figures/confusion_matrix.svg"))
display(SVG(f"{results_dir}/figures/training_curves.svg"))

In [ ]:
# SHAP feature importance for the predicted faulty probability.
%pip install shap -q
import shap

background_size = min(50, len(train_idx))
explain_size = min(100, len(test_X))
background = features[train_idx[:background_size]]
explain_X = test_X[:explain_size]

def predict_faulty_probability(values):
    values = np.asarray(values, dtype=np.float64)
    return model.predict_proba(values)[:, 1]

explainer = shap.Explainer(
    predict_faulty_probability,
    background,
    feature_names=feature_names,
)
shap_explanation = explainer(explain_X)
shap_values = np.asarray(shap_explanation.values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.mean(np.abs(shap_values), axis=0),
}).sort_values("mean_abs_shap", ascending=False)
print(shap_importance.head(20))
shap_importance.to_csv(f"{results_dir}/shap_feature_importance.csv", index=False)

shap.summary_plot(
    shap_values,
    explain_X,
    feature_names=feature_names,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(f"{results_dir}/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()